# Classical Text Processing and Modern LLM Subword Segmentation

## Import All Libraries

In [ ]:
import re
import os
from dotenv import load_dotenv
from tokenizers import Tokenizer
from datasets import load_dataset
from huggingface_hub import login
from tokenizers.models import BPE, WordPiece
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer, WordPieceTrainer


load_dotenv()
login(os.getenv("HUGGINGFACE_TOKEN"))

## 1. Email Validation and Parsing using Regular Expression

This section defines regular expressions to validate email addresses based on specific constraints (e.g., proper use of '@', valid characters in the local-part, and correct domain formatting). It also implements a parser to extract and validate names and emails from a provided text file, ensuring that the local-part of the email corresponds to the person's registered name.

In [1]:
line_pattern = re.compile(r"name=([^,]+), email=([^\s,]+)")
email_pattern = re.compile(r"^[a-zA-Z0-9_-]+(?:\.[a-zA-Z0-9_-]+)*@(?:[a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}$")

def is_name_email_match(name, email):
    parts = name.lower().split()
    if len(parts) < 2:
        return False

    first, last = parts[0], parts[-1]
    local = email.split("@", 1)[0].lower()
    pattern = (
        rf"(?:{re.escape(first)}[._-]?{re.escape(last)}|"
        rf"{re.escape(last)}[._-]?{re.escape(first)}|"
        rf"{re.escape(first[0])}[._-]?{re.escape(last)})"
    )

    return re.fullmatch(pattern, local) is not None

def parse_and_validate_emails(file_path):
    valid_records = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            match = line_pattern.search(line.strip())

            if not match:
                continue

            name, email = match.group(1).strip(), match.group(2).strip()

            if email_pattern.match(email) and is_name_email_match(name, email):
                valid_records.append({"name": name, "email": email})

    return valid_records

def find_email_by_person_name(name, valid_records):
    parts = name.lower().split()

    if len(parts) < 2:
        return f"Invalid query name: {name}"

    first, last = parts[0], parts[-1]
    pattern = rf"(?:{re.escape(first)}[._-]?{re.escape(last)}|{re.escape(last)}[._-]?{re.escape(first)}|{re.escape(first[0])}[._-]?{re.escape(last)})"

    for record in valid_records:
        if re.search(pattern, record['email'].lower()):
            return record['email']

    return f"No matching valid email found for {name}"


valid_emails = parse_and_validate_emails('emails.txt')
print(f"Number of valid emails: {len(valid_emails)}\n")
for record in valid_emails:
    print(f"Name: {record['name']}, Email: {record['email']}")

Number of valid emails: 22

Name: Milad Fattahi, Email: milad.fattahi@uni.ir
Name: Parsa Golkar, Email: parsa.golkar@host.ir
Name: Reza Moradi, Email: moradi.reza@uni.ir
Name: Niloufar Heidari, Email: niloufarheidari@lab.ir
Name: Mahsa Akbari, Email: mahsa_akbari@domain.com
Name: Ali Rezaei, Email: ali.rezaei@example.com
Name: Farhad Zarei, Email: zarei-farhad@host.ir
Name: Hamed Jalali, Email: hamed.jalali@net.com
Name: Ramin Shafiei, Email: ramin.shafiei@project.net
Name: Laleh Sadeghi, Email: sadeghi.laleh@work.ir
Name: Pouya Nouri, Email: nouri.pouya@mail.ir
Name: Elham Davari, Email: elhamdavari@office.net
Name: Behnam Khatibi, Email: behnam.khatibi@mail.org
Name: Sara Soleimani, Email: sara.soleimani@host.ir
Name: Shirin Hashemi, Email: shirin_hashemi@domain.com
Name: Arman Khalili, Email: arman.khalili@school.ir
Name: Raha Esfandiari, Email: rahaesfandiari@company.com
Name: Kiana Ghasemi, Email: kiana.ghasemi@uni.edu
Name: Navid Khodadadi, Email: navid_khodadadi@research.org
Nam

## 2. Querying Validated Emails by Name

Here, we define a search function to find the valid email address associated with a specific person's name from our parsed records. We then test this function using the query names 'Behnam Khatibi' and 'Mehrdad Ebrahimi'.

In [2]:
def find_email_by_person_name(name, valid_records):
    parts = name.lower().split()

    if len(parts) < 2:
        return f"Invalid query name: {name}"

    first, last = parts[0], parts[-1]
    pattern = rf"(?:{re.escape(first)}[._-]?{re.escape(last)}|{re.escape(last)}[._-]?{re.escape(first)}|{re.escape(first[0])}[._-]?{re.escape(last)})"

    for record in valid_records:
        if re.search(pattern, record['email'].lower()):
            return record['email']

    return f"No matching valid email found for {name}"

print(find_email_by_person_name("Behnam Khatibi", valid_emails))
print(find_email_by_person_name("Mehrdad Ebrahimi", valid_emails))

behnam.khatibi@mail.org
mehrdad.ebrahimi@example.org


## 3. Levenshtein Distance (Minimum Edit Distance) Implementation

This cell implements the Levenshtein distance algorithm utilizing dynamic programming to calculate the minimum edit distance between two strings. The algorithm then evaluates specific word pairs to measure the cost of insertions, deletions, and substitutions.

In [3]:
def levenshtein_distance(str1, str2, sub_cost=1):
    m, n = len(str1), len(str2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if str1[i - 1] == str2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = min(
                    dp[i - 1][j] + 1,      # Deletion
                    dp[i][j - 1] + 1,      # Insertion
                    dp[i - 1][j - 1] + sub_cost # Substitution
                )
    return dp[m][n]

def auto_correct_sentence(sentence, vocabulary):
    tokens = sentence.split()
    corrected = []
    vocab_set = set(vocabulary)

    for token in tokens:
        match = re.match(r"^(\w+)([^\w]*)$", token)
        if not match:
            corrected.append(token)
            continue

        word, punct = match.group(1), match.group(2)
        word_lower = word.lower()

        if word_lower in vocab_set or len(word_lower) <= 2:
            corrected.append(word + punct)
        else:
            best_candidate = min(vocabulary, key=lambda v: levenshtein_distance(word_lower, v.lower()))
            corrected.append(best_candidate + punct)

    return " ".join(corrected)

# Test pairs
pairs = [("Athletic", "Atlantic"), ("London", "Boston"), ("Action", "Compact"), ("", "Sting")]
print("Levenshtein Distances (insert cost = 1, delete cost = 1, substitute cost = 2):\n")
for w1, w2 in pairs:
    print(f"Distance ('{w1}', '{w2}') = {levenshtein_distance(w1, w2, sub_cost=2)}")

Levenshtein Distances (insert cost = 1, delete cost = 1, substitute cost = 2):

Distance ('Athletic', 'Atlantic') = 4
Distance ('London', 'Boston') = 6
Distance ('Action', 'Compact') = 9
Distance ('', 'Sting') = 5


## 4. Sentence Spell Correction using Edit Distance

Using the previously implemented Levenshtein distance function, this script corrects misspelled words in a target sentence. It cross-references each word against a provided vocabulary list and replaces incorrect words with the closest valid match found via minimum edit distance.

In [4]:
sentence = "helo studnts at the universty are wrting ther frst edit distnce algorthm in pythn, and they reely enjy it!"

with open("vocab.txt", "r", encoding="utf-8") as f:
    vocabs = f.read().split(", ")

words = sentence.split()
corrected_words = []

for word in words:
    # Assume 2-letter and 3-letter words are not misspelled
    if word in vocabs or len(word) <= 3:
        corrected_words.append(word)
    else:
        min_distance = float('inf')
        closest_vocab = None
        for vocab in vocabs:
            if len(word)+1 == len(vocab) or len(word)-1 == len(vocab):
                distance = levenshtein_distance(word, vocab)
                if distance < min_distance:
                    min_distance = distance
                    closest_vocab = vocab
            else:
                continue

            distance = levenshtein_distance(word, vocab)
            if distance < min_distance:
                min_distance = distance
                closest_vocab = vocab
        corrected_words.append(closest_vocab)

corrected_sentence = " ".join(corrected_words)

print(f"Original Sentence: {sentence}\n")
print(f"Corrected Sentence: {corrected_sentence}")

Original Sentence: helo studnts at the universty are wrting ther frst edit distnce algorthm in pythn, and they reely enjy it!

Corrected Sentence: hello students at the university are writing their first edit distance algorithm in learn and they really enjoy it!


## 5. Regex-Based Text Tokenization and Improvement

This section introduces a basic regex tokenizer that splits text based on standard punctuation and whitespace. Since the basic version struggles with complex structures like abbreviations (e.g., U.S.A.) and currencies, an improved tokenizer is proposed and implemented to handle these edge cases effectively using a verbose regex pattern.

In [5]:
def basic_regex_tokenizer(text):
    pattern = r"\w+|[^.,;:?!()\s]"
    return re.findall(pattern, text)

def improved_regex_tokenizer(text):
    pattern = r"[A-Za-z]+(?:[-.'][A-Za-z]+)*|\.\.\.+|\$?\d+(?:\.\d+)?%?|[.!?,;:()\"']"
    return re.findall(pattern, text, flags=re.VERBOSE)

sample_sentences = [
    "Hello, world! NLP is fun.",
    "That U.S.A. poster-print costs $12.40..."
]

for sent in sample_sentences:
    print(f"Input: {sent}")
    print(f"Basic Tokenizer ({len(basic_regex_tokenizer(sent))} tokens): {basic_regex_tokenizer(sent)}")
    print(f"Improved Tokenizer ({len(improved_regex_tokenizer(sent))} tokens): {improved_regex_tokenizer(sent)}\n")

Input: Hello, world! NLP is fun.
Basic Tokenizer (5 tokens): ['Hello', 'world', 'NLP', 'is', 'fun']
Improved Tokenizer (8 tokens): ['Hello', ',', 'world', '!', 'NLP', 'is', 'fun', '.']

Input: That U.S.A. poster-print costs $12.40...
Basic Tokenizer (11 tokens): ['That', 'U', 'S', 'A', 'poster', '-', 'print', 'costs', '$', '12', '40']
Improved Tokenizer (7 tokens): ['That', 'U.S.A', '.', 'poster-print', 'costs', '$12.40', '...']



## 6. Training BPE and WordPiece Tokenizers on a Persian Dataset

In this final step, we load the `TinyStories-Farsi` dataset from HuggingFace to train custom tokenizers. We train both a Byte-Pair Encoding (BPE) tokenizer and a WordPiece tokenizer on the Persian text corpus. We then evaluate and compare their tokenization strategies on a sample Persian sentence.

In [6]:
# Load dataset
raw_dataset = load_dataset("taesiri/TinyStories-Farsi", split="train")
corpus = [item["Persian"] for item in raw_dataset if item["Persian"]]

# 1. Train BPE Tokenizer
bpe_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
bpe_tokenizer.pre_tokenizer = Whitespace()
bpe_trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"], vocab_size=8000)
bpe_tokenizer.train_from_iterator(corpus, trainer=bpe_trainer)

# 2. Train WordPiece Tokenizer
wp_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
wp_tokenizer.pre_tokenizer = Whitespace()
wp_trainer = WordPieceTrainer(special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"], vocab_size=8000)
wp_tokenizer.train_from_iterator(corpus, trainer=wp_trainer)

# Test sentence
test_text = "روزی یک مرد ثروتمند، پسر بچه کوچکش را بـه ده برد تا بـه او نشان دهد مردمی که در آنجا زندگی می‌کنند، چقدر فقیر هستند."

print(f"BPE Tokens ({len(bpe_tokenizer.encode(test_text).tokens)}): {bpe_tokenizer.encode(test_text).tokens}\n")
print(f"WordPiece Tokens ({len(wp_tokenizer.encode(test_text).tokens)}): {wp_tokenizer.encode(test_text).tokens}")

BPE Tokens (32): ['روزی', 'یک', 'مرد', 'ثروتمند', '،', 'پسر', 'بچه', 'کوچکش', 'را', 'ب', 'ـ', 'ه', 'ده', 'برد', 'تا', 'ب', 'ـ', 'ه', 'او', 'نشان', 'دهد', 'مردمی', 'که', 'در', 'آنجا', 'زندگی', 'می\u200cکنند', '،', 'چقدر', 'فقیر', 'هستند', '.']

WordPiece Tokens (32): ['روزی', 'یک', 'مرد', 'ثروتمند', '،', 'پسر', 'بچه', 'کوچکش', 'را', 'ب', '##ـ', '##ه', 'ده', 'برد', 'تا', 'ب', '##ـ', '##ه', 'او', 'نشان', 'دهد', 'مردمی', 'که', 'در', 'آنجا', 'زندگی', 'می\u200cکنند', '،', 'چقدر', 'فقیر', 'هستند', '.']
